# Vietlott Data Science & Machine Learning Exploration
Dự án phân tích thống kê, kiểm định tính ngẫu nhiên và backtesting chiến lược dự đoán cho **Power 6/55** và **Mega 6/45**.

> **Lưu ý**: Xổ số là sự kiện ngẫu nhiên độc lập. Notebook này phục vụ mục đích nghiên cứu Data Science & Backtest, không cam kết khả năng trúng giải.

In [ ]:
# 1. Cài đặt thư viện (cho Google Colab / Môi trường mới)
# !pip install pandas numpy scipy scikit-learn matplotlib seaborn pyyaml requests
import os
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import VietlottDataLoader
from src.analyzer import VietlottAnalyzer
from src.randomness_test import VietlottRandomnessTester
from src.strategies import (
    RandomBaselineStrategy, HotFrequencyStrategy, ColdOverdueStrategy,
    HybridStrategy, MLRankingStrategy
)
from src.backtester import VietlottBacktester
from src.visualizer import VietlottVisualizer

print("Thư viện đã được tải thành công!")

## 2. Tải và làm sạch dữ liệu
Chọn sản phẩm xổ số:  hoặc .

In [ ]:
LOTTERY = "power655"  # hoặc "mega645"
TOTAL_NUMBERS = 55 if LOTTERY == "power655" else 45
PICK_COUNT = 6

loader = VietlottDataLoader(lottery_code=LOTTERY, raw_dir="../data/raw", processed_dir="../data/processed")
df = loader.load_clean_data(force_refresh=False)
print(f"Đã tải {len(df):,} kỳ quay. Từ {df['date'].min().date()} đến {df['date'].max().date()}.")
df.tail(5)

## 3. Thống kê mô tả & Phân tích số Nóng / Lạnh / Khoảng cách (Gaps)

In [ ]:
analyzer = VietlottAnalyzer(df, total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT)
hot_cold = analyzer.get_hot_cold_numbers(lookback=50, top_n=10)
print("=== TOP 5 SỐ NÓNG (Gần đây nhất) ===")
print(pd.DataFrame(hot_cold["hot_numbers"]).head(5))

print("
=== TOP 5 SỐ LÂU CHƯA VỀ (Overdue Gaps) ===")
print(pd.DataFrame(hot_cold["overdue_numbers"]).head(5))

## 4. Kiểm định tính ngẫu nhiên (Randomness Hypothesis Testing)

In [ ]:
tester = VietlottRandomnessTester(df, total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT)
test_res = tester.run_all_tests()
print("1. Chi-square Uniformity Test:", test_res["uniformity_test"]["conclusion"])
print(f"   chi2={test_res['uniformity_test']['chi2_stat']}, p-value={test_res['uniformity_test']['p_value']}")
print("2. Runs Test for Independence:", test_res["runs_test"]["conclusion"])
print("3. Repeat Hypergeometric Test:", test_res["repeat_test"]["conclusion"])

## 5. Walk-Forward Backtesting các chiến lược

In [ ]:
strategies = [
    RandomBaselineStrategy(total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT, random_state=42),
    HotFrequencyStrategy(total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT, lookback_window=50, random_state=42),
    ColdOverdueStrategy(total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT, random_state=42),
    HybridStrategy(total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT, lookback_window=50, random_state=42),
    MLRankingStrategy(total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT, train_window=200, model_type="logreg", random_state=42)
]

backtester = VietlottBacktester(df, lottery_code=LOTTERY, total_numbers=TOTAL_NUMBERS, pick_count=PICK_COUNT)
bt_res = backtester.run_backtest(strategies, test_draws=150, tickets_per_draw=5)

print("=== BẢNG KẾT QUẢ BACKTEST ===")
bt_res["summary_df"][["strategy", "total_tickets", "avg_hits_per_ticket", "theoretical_hits", "roi_pct", "win_rate_pct", "p_value_vs_random"]]

## 6. Gợi ý bộ số cho kỳ quay tiếp theo

In [ ]:
print("=== GỢI Ý BỘ SỐ KỲ TIẾP THEO (5 VÉ/CHIẾN LƯỢC) ===")
for strat in strategies:
    tickets = strat.generate_tickets(df, num_tickets=5)
    print(f"
-- {strat.name} --")
    for idx, t in enumerate(tickets, 1):
        print(f"  Vé #{idx}: {t} | Tổng: {sum(t)}")